In [6]:
%reset -f

Loading RAPIDS Packages

In [1]:
import cudf as pd
import cupy as np
import numpy as np_standard
from cuml.model_selection import train_test_split
from cuml.ensemble import RandomForestClassifier
from cuml.linear_model import LogisticRegression
from cuml.svm import SVC
from cuml.naive_bayes import GaussianNB
from cuml.neighbors import KNeighborsClassifier

# Additional RAPIDS-compatible packages
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score as sk_accuracy_score
from sklearn.metrics import classification_report as sk_classification_report

In [2]:
# Try to import additional tree models
try:
    from sklearn.ensemble import BaggingClassifier as BART  # Approximation of BART
except ImportError:
    BART = None

try:
    from sklearn.ensemble import GradientBoostingClassifier
except ImportError:
    GradientBoostingClassifier = None

# Try to import BART specifically
try:
    from bartpy.sklearnmodel import SklearnModel as BARTModel

    BART_AVAILABLE = True
except ImportError:
    BART_AVAILABLE = False
    print("BART not available, using BaggingClassifier as approximation")

import warnings

warnings.filterwarnings('ignore')

print("RAPIDS packages loaded successfully")

BART not available, using BaggingClassifier as approximation
RAPIDS packages loaded successfully


Load and Preprocess Data

In [3]:
dataset = pd.read_parquet('../../TabulatedData/values_all_layers_16bit.parquet')

X = dataset.drop("name", axis=1)
y = dataset["name"]

# OneHot Encoding for target variable
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
y_numpy = y.to_numpy()  # Convert cudf Series to numpy array
y_one_hot = encoder.fit_transform(y_numpy.reshape(-1, 1))
y_encoded = np_standard.argmax(y_one_hot, axis=1)

# Convert back to cudf for compatibility with RAPIDS
y_encoded = pd.Series(y_encoded)

# Get class names for reporting
class_names = encoder.categories_[0]

print("Data Loaded and Encoded")
print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(class_names)}")

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.33, random_state=42)

del dataset, X, y, y_encoded
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")


Data Loaded and Encoded
Dataset shape: (22130, 25610)
Number of classes: 206
Training set shape: (14827, 25610)
Test set shape: (7303, 25610)


Define Models Dictionary

In [5]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(objective='multi:softmax', num_class=len(class_names), random_state=42),
    'LightGBM': lgb.LGBMClassifier(objective='multiclass', num_class=len(class_names), random_state=42),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, verbose=False, random_state=42),
    'Bagging': BaggingClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(),
    'SVM': SVC(random_state=42),
    'Naive Bayes': GaussianNB(),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

# Add additional requested models
if BART_AVAILABLE:
    models['BART'] = BARTModel(n_trees=50, n_chains=4, n_samples=200, n_burn=100, thin=10, random_state=42)
else:
    models['BART (Approx)'] = BaggingClassifier(n_estimators=50, random_state=42)

# Model-Based Trees (using Gradient Boosting as approximation)
models['Model-Based Trees'] = GradientBoostingClassifier(n_estimators=100, random_state=42)

# Conditional Inference Trees (using Decision Tree with specific parameters)
models['Conditional Inference Trees'] = DecisionTreeClassifier(criterion='entropy', splitter='best', max_depth=10,
                                                               random_state=42)


Train and Evaluate Models

In [ ]:
results = {}

for name, model in models.items():
    print(f"\n{'=' * 50}")
    print(f"Training {name}...")
    print(f"{'=' * 50}")

    try:
        # Convert cudf data to numpy for sklearn models
        X_train_np = X_train.to_numpy() if hasattr(X_train, 'to_numpy') else X_train.values if hasattr(X_train,
                                                                                                       'values') else X_train
        y_train_np = y_train.to_numpy() if hasattr(y_train, 'to_numpy') else y_train.values if hasattr(y_train,
                                                                                                       'values') else y_train
        X_test_np = X_test.to_numpy() if hasattr(X_test, 'to_numpy') else X_test.values if hasattr(X_test,
                                                                                                   'values') else X_test
        y_test_np = y_test.to_numpy() if hasattr(y_test, 'to_numpy') else y_test.values if hasattr(y_test,
                                                                                                   'values') else y_test

        # Train model
        model.fit(X_train_np, y_train_np)

        # Make predictions
        y_pred_train = model.predict(X_train_np)
        y_pred_test = model.predict(X_test_np)

        # Calculate metrics
        train_acc = sk_accuracy_score(y_train_np, y_pred_train)
        test_acc = sk_accuracy_score(y_test_np, y_pred_test)

        results[name] = {
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'model': model
        }

        print(f"{name} Results:")
        print(f"Training Accuracy: {train_acc:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}")

        # Print detailed classification report for test set
        print("\nDetailed Classification Report (Test Set):")
        print(sk_classification_report(
            y_test_np,
            y_pred_test,
            target_names=class_names,
            zero_division=0
        ))

    except Exception as e:
        print(f"Error training {name}: {str(e)}")
        results[name] = {'error': str(e)}



Training Decision Tree...
Decision Tree Results:
Training Accuracy: 0.9869
Test Accuracy: 0.1735

Detailed Classification Report (Test Set):
                    precision    recall  f1-score   support

      Aditya Kundu       0.35      0.30      0.32        20
       Akash Gupta       0.40      0.32      0.36        59
          Ankur De       0.54      0.50      0.52       187
           Ashtavi       0.19      0.22      0.21        32
          Avyuktha       0.09      0.13      0.11       105
        Harshith H       0.39      0.43      0.41        30
     Jiya Sachdeva       0.29      0.24      0.27        94
     Karthikeya SK       0.17      0.20      0.18        20
    Nandini Sharma       0.29      0.33      0.31        49
           Navnita       0.05      0.06      0.05       104
          Papia De       0.60      0.53      0.56        47
        Pavithra S       0.53      0.48      0.51        95
            Piyali       0.26      0.36      0.30        22
Prajwal Mundigana

Summary Results

In [ ]:
print("\n" + "=" * 80)
print("SUMMARY OF RESULTS")
print("=" * 80)

summary_data = []
for name, result in results.items():
    if 'error' not in result:
        summary_data.append({
            'Model': name,
            'Train Accuracy': f"{result['train_accuracy']:.4f}",
            'Test Accuracy': f"{result['test_accuracy']:.4f}"
        })
    else:
        summary_data.append({
            'Model': name,
            'Train Accuracy': 'Error',
            'Test Accuracy': result['error']
        })

# Create summary DataFrame
import pandas as pd

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Find best performing model
valid_results = {k: v for k, v in results.items() if 'error' not in v}
if valid_results:
    best_model = max(valid_results.items(), key=lambda x: x[1]['test_accuracy'])
    print(f"\n{'=' * 50}")
    print(f"Best Performing Model: {best_model[0]}")
    print(f"Test Accuracy: {best_model[1]['test_accuracy']:.4f}")
    print(f"{'=' * 50}")

print("\nAll models evaluated successfully!")
